# 02 — Clean & Derive language metrics

Load `speeches_raw` from DuckDB, clean it, and derive the per-speech metrics the
lead question needs. Output: `data/interim/speeches_clean.parquet` + DuckDB tables
`speeches_clean`, `president_terms`, `word_freq_by_president`,
`distinctive_words_by_president`.

**Lead question:** self (I/me/my) vs collective (we/us/our) framing by president, and
the 1789→present trend. Plus (owner ask, 2026-09-25): straight **word frequencies**
for word clouds, **common vs distinctive** words per president, and a way to **weight
by presidency length** (4 vs 8-yr terms, partial terms).

## ⭐ Guarding against apples-to-oranges

The corpus is **curated** and coverage is far denser for modern presidents (LBJ 71
speeches … Garfield 1). So we derive **`speech_type`** so per-president comparisons are
made within a lens (whole corpus = context only; **SOTU series** = Annual Message +
State of the Union unified; **Inaugural**). Even within SOTU there's a **written
(clerk-read, 1801–1912) vs spoken** break flagged as `delivery_mode`.

Metric + text logic lives in `src/clean_quality.py`. Tokenization is a transparent
lowercase word regex that first HTML-unescapes the source (so `&ldquo;`/`&mdash;`
fragments don't leak in as fake words) and attributes contractions (I'm/we'll/let's)
to their pronoun — every count is codebook-explainable, no hidden NLP model.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_interim,
    load_to_duckdb, register_source, add_speech_metrics,
    top_words_by_group, distinctive_words_by_group, stopwords,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Load raw + parse date/year in DuckDB

Keep only the columns we use. The `date` field is ISO with a bogus fixed `-04:56`
offset on every row — an artifact — so we take the leading `YYYY-MM-DD` and derive `year`.

In [ ]:
df = run_sql("""
    SELECT
        TRIM(president)              AS president,
        CAST(date[1:10] AS DATE)     AS speech_date,
        CAST(date[1:4]  AS INTEGER)  AS year,
        TRIM(title)                  AS title,
        transcript,
        url,
        source_file
    FROM speeches_raw
    WHERE transcript IS NOT NULL AND TRIM(transcript) <> ''
""", con)
print(df.shape)
print('year span:', int(df.year.min()), '→', int(df.year.max()))
df.head(3)

## Derive per-speech language metrics + speech type

`add_speech_metrics()` adds: `word_count`, `self_count`, `collective_count`,
`self_per_1k`, `collective_per_1k`, `self_share`, `speech_type`, `is_sotu_series`.
Rates are per 1,000 words so speeches of different lengths compare fairly.

In [ ]:
df = add_speech_metrics(df, text_col='transcript', title_col='title')
df['delivery_mode'] = df['year'].map(lambda y: 'written_era' if 1801 <= y <= 1912 else 'spoken_era')
df[['president','year','speech_type','word_count','self_count','collective_count',
    'self_per_1k','collective_per_1k','self_share']].head(6)

## Sanity-check the classification + the three lenses

In [ ]:
print('=== speech_type counts ===')
print(df['speech_type'].value_counts().to_string())
print('\n=== coverage by lens ===')
for lens, mask in {
    'Whole corpus':      df['president'].notna(),
    'SOTU series':       df['is_sotu_series'],
    'Inaugural Address': df['speech_type'] == 'Inaugural Address',
}.items():
    sub = df[mask]
    print(f"{lens:20s} n={len(sub):4d}  presidents={sub['president'].nunique():3d}  "
          f"{int(sub.year.min())}–{int(sub.year.max())}")

## President term lengths — to weight by tenure (owner ask)

To fairly compare presidents we need to control for time in office: an 8-year, a
4-year, and a 31-day presidency (W. Harrison) are not comparable on raw speech counts.
We load a curated `data/raw/president_terms.csv` of inauguration/exit dates (uncontested
public record) and compute **days in office**. Non-consecutive presidencies (Grover
Cleveland; Donald Trump) have one row per continuous span, summed — so they don't
absorb the intervening president's time. Trump's second term is in progress, capped at
the data retrieval date (days accrued so far) — a caveat for any per-year rate.

This gives `speeches_per_year` = speeches ÷ (days_in_office / 365.25), the tenure-
weighted volume metric.

In [ ]:
terms_raw = pd.read_csv('data/raw/president_terms.csv', parse_dates=['term_start','term_end'])
terms_raw['days'] = (terms_raw['term_end'] - terms_raw['term_start']).dt.days
terms = (terms_raw.groupby('president', as_index=False)['days'].sum()
         .rename(columns={'days': 'days_in_office'}))
terms['years_in_office'] = (terms['days_in_office'] / 365.25).round(2)

# speeches in the CURATED corpus per president (a coverage measure, not total output)
n_speeches = df.groupby('president', as_index=False).size().rename(columns={'size': 'n_speeches'})
terms = terms.merge(n_speeches, on='president', how='left')
terms['n_speeches'] = terms['n_speeches'].fillna(0).astype(int)
terms['speeches_per_year'] = (terms['n_speeches'] / terms['years_in_office']).round(2)
# short-tenure guard: a per-year RATE off <1yr in office (W. Harrison 31 days,
# Garfield ~200 days) is a tiny-denominator artifact — flag it so charts can drop
# or asterisk those presidents rather than rank them as if comparable.
terms['reliable_rate'] = terms['years_in_office'] >= 1.0

# name-match guard: every president in the data must be in the term table
missing = set(df.president) - set(terms.president)
assert not missing, f'presidents missing from term table: {missing}'
print('term table:', len(terms), 'presidents; 0 name mismatches')
terms.sort_values('years_in_office').head(4)

In [ ]:
load_to_duckdb(terms, 'president_terms', con)
register_source(
    con, table='president_terms',
    name='U.S. presidential term dates (curated public record)',
    url='https://data.millercenter.org/  (dates cross-checked against public record)',
    license='Public domain (historical fact)',
    notes=('Inauguration/exit dates per president, days_in_office computed. '
           'Non-consecutive terms (Cleveland, Trump) summed across continuous spans. '
           'Trump 2nd term in progress, capped at data retrieval date.'),
    methodology='Hand-curated from uncontested public record (inauguration + exit dates).',
    series_breaks='Trump 2nd-term days are partial (through retrieval date) — per-year rate is provisional.',
)
print('president_terms rows:', con.execute('SELECT COUNT(*) FROM president_terms').fetchone()[0])

## Word frequencies — common words (per president) for word clouds

Content-word frequencies per president (transparent stopword list removed — function
words, the I/we pronoun set counted separately, and a little speech boilerplate; see
`stopwords()`). `word_freq_by_president` (top 60/president) feeds the per-president
word-cloud picker. These are the words each president says a lot — which is partly just
what every president says a lot (people, nation, states).

In [ ]:
print('stopwords in list:', len(stopwords()))
word_freq = top_words_by_group(df, 'president', 'transcript', top_n=60)
load_to_duckdb(word_freq, 'word_freq_by_president', con)
print('word_freq_by_president rows:', len(word_freq))
print('\nLincoln top 10 (common):',
      word_freq[word_freq.president=='Abraham Lincoln'].head(10).word.tolist())

## Distinctive words — TF-IDF (what DEFINES each president)

The more interesting cut: words common *for this president* but rare *across all
presidents* — their defining vocabulary. Each president = one document (all their
speeches); score = term-freq (per 10k content words) × log(N_presidents / N using the
word). Pure-Python, inspectable, no sklearn. `distinctive_words_by_president`.

In [ ]:
distinctive = distinctive_words_by_group(df, 'president', 'transcript', top_n=60)
load_to_duckdb(distinctive, 'distinctive_words_by_president', con)
print('distinctive_words_by_president rows:', len(distinctive))
for p in ['Abraham Lincoln','Franklin D. Roosevelt','Ronald Reagan','Barack Obama']:
    print(f"{p:24s}", distinctive[distinctive.president==p].head(6).word.tolist())

## Quality report + save

In [ ]:
qr = quality_report(
    df, table_name='speeches_clean', con=con,
    required_columns=['president','year','speech_type','word_count',
                      'self_count','collective_count'],
    max_null_pct=0.05,
)
print('\nspeeches with 0 self+collective pronouns (self_share null):',
      int((df.self_count + df.collective_count == 0).sum()))

In [ ]:
load_to_duckdb(df, 'speeches_clean', con)
save_interim(df, cfg, 'speeches_clean.parquet')
# drop the quality-report scratch table so it doesn't persist in the DB
con.execute('DROP TABLE IF EXISTS _qc_speeches_clean')
print('speeches_clean rows in DuckDB:',
      con.execute('SELECT COUNT(*) FROM speeches_clean').fetchone()[0])
print('tables now:', [r[0] for r in con.execute('SHOW TABLES').fetchall()])

---
**Next:** `04-viz.ipynb` — explore self/collective (3 lenses), word-count angles,
common vs distinctive words, tenure-weighted speech volume, and the per-president
word-cloud picker. Pause for owner review before `06-viz-social`.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')